In [1]:
# Cryptogram Project
# Author: Althea Doherty
# Description: creating a cryptogram Puzzle

In [5]:
import random
import tkinter as tk

#Variables
PUZLVL = {
    "0: Rules":[],
    "1: Easy": [ ("", ""),("", "") ],
    "2: Medium": [ ("", ""), ("", "") ],
    "3: Hard": [   ("", ""),  ("", "") ]}

#Main Loop
window = tk.Tk()
window.geometry("850x550")
window.title("Cryptogram Puzzle")

health = tk.IntVar(value=100)
lvlselect = tk.StringVar(value="0: Rules")  # Default difficulty

# Global game state variables
correct = ""
encrypted = ""
entries = []        # Stores tuple: (ogIndx, eWidget)
corAns = []         # Stores list of uppercase letters in correct quote

#Window Set Up Functions & Validations
def validChar(newVal):
    if newVal == "":
        return True
    return len(newVal) == 1 and newVal.isalpha()

vcmd = (window.register(validChar), "%P")

def hLvl(amount):
    hupdate = max(0, health.get() - amount)
    health.set(hupdate)
    hLabel.config(text=f"Health: {hupdate}")

def loadLevel(key):
    global correct, encrypted, corAns, entries

    # Reset health
    health.set(100)
    hLabel.config(text="Health: 100")

    # Clear previous widgets in Puzzle and history frames
    for widget in pFrame.winfo_children():
        widget.destroy()
    for widget in gFrame.winfo_children():
        widget.destroy()

    #setting default screen to open to the rules
    if key =="0: Rules":
        hLabel.config(text="") #hiding the health bar when rules open
        subButton.config(state=tk.DISABLED) #turn off the submission button when rules open

        rulestxt = ("...")
        tk.Label(pFrame, text=rulestxt, font=("Times",14), justify="left").pack(pady=20)
        return

    #Re-enable the submit button and health on the Tinker Window
    subButton.config(state=tk.NORMAL)
    health.set(100)
    hLabel.config(text="Health: 100")
    
    # Select random quote pair based on difficulty (correct,encrypted)
    cQuote, eQuote = random.choice(PUZLVL[key])
    correct = cQuote.upper()
    encrypted = eQuote.upper()

    corAns = [ch for ch in correct if ch.isalpha()]
    entries.clear()

    # Build Puzzle grid dynamically
    for i, ch in enumerate(encrypted):
        label = tk.Label(pFrame, text=ch, font=("Great Vibes", 14, "bold"))
        label.grid(row=0, column=i, padx=3)

        if ch.isalpha():
            entry = tk.Entry(pFrame, width=3, font=("Terminal", 12), justify="center", validate="key", validatecommand=vcmd)
            entry.grid(row=1, column=i, padx=3)
            # Store tuple of (string_index, widget) for accurate checking
            entries.append((i, entry))
        else:
            tk.Label(pFrame, text=ch, font=("Times", 14)).grid(row=1, column=i)

#User Guesses
def subGuess():
    wrong = False
    correct_guesses = {}

    # Check each entry box against its exact string position
    for ogIndx, eWidget in entries:
        guess = eWidget.get().upper()
        reLet = correct[ogIndx]

        if guess == reLet:
            eWidget.config(bg="OliveDrab1")
            if guess:
                correct_guesses[reLet] = guess
        else:
            eWidget.config(bg="tomato")
            if guess:
                wrong = True

    # Deduct health if any wrong guess was made
    if wrong:
        hLvl(10)

    # Autofill matching letters in all positions
    for reLet, uGuess in correct_guesses.items():
        for ogIndx, eWidget in entries:
            if correct[ogIndx] == reLet:
                eWidget.delete(0, tk.END)
                eWidget.insert(0, uGuess)
                eWidget.config(bg="OliveDrab1")

    # Record history row (using sequential index for neat alignment)
    row = tk.Frame(gFrame)
    row.pack(pady=2)
    for col_idx, (ogIndx, eWidget) in enumerate(entries):
        tk.Label(row, text=eWidget.get().upper() or "-", width=3, font=("Terminal", 11), justify="center", bg=eWidget.cget("bg")).grid(row=0, column=col_idx, padx=2)

def restartGame():
    loadLevel(lvlselect.get())

#User Select Menu Design
menuFrame = tk.Frame(window)
menuFrame.pack(pady=5)

mLabel = tk.Label(menuFrame, text="Difficulty: ", font=("Great Vibes", 10, "bold"))
mLabel.pack(side=tk.LEFT, padx=5) #Aligning Text Left

# Dropdown OptionMenu replacing console while loop
#Traditional Input Menu was not working, would stall/halt program for user input & tinker already had a solution
lvlDrop = tk.OptionMenu(menuFrame, lvlselect, *PUZLVL.keys(), command=loadLevel)
lvlDrop.pack(side=tk.LEFT)

# Health & Headers
hLabel = tk.Label(window, text="Health: 100", font=("Jokerman", 12, "bold"))
hLabel.pack(pady=5)

eLabel = tk.Label(window, text="Cryptogram Board", font=("Great Vibes", 11, "italic"))
eLabel.pack(pady=5)

pFrame = tk.Frame(window)
pFrame.pack(pady=10)

# Action Buttons
btnFrame = tk.Frame(window)
btnFrame.pack(pady=5)

subButton = tk.Button(btnFrame, text="Submit Entry", command=subGuess)
subButton.pack(side=tk.LEFT, padx=5)

restartButton = tk.Button(btnFrame, text="Restart Level", command=restartGame)
restartButton.pack(side=tk.LEFT, padx=5)

gFrame = tk.Frame(window)
gFrame.pack(pady=10)

# Initialize game with default level (0: Rules)
loadLevel(lvlselect.get())

window.mainloop()